# **Forçar montar novamente o acesso.**

In [7]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re

# ====================================================================
# 1. Montagem, Caminhos e Limpeza (MODIFICADO)
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    # Assumindo que o drive.mount foi executado com sucesso em uma célula anterior
    pass
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# --- Solicita o nome do arquivo de entrada (REVERTIDO PARA UM ÚNICO INPUT) ---
NOME_ARQUIVO_TXT = input("Digite o nome do arquivo TXT para análise (Ex: PicoW.txt, S1.txt): ").strip()
if not (NOME_ARQUIVO_TXT.endswith('.txt') or NOME_ARQUIVO_TXT.endswith('.csv')):
    NOME_ARQUIVO_TXT += '.txt'

ARQUIVO_ENTRADA_TXT = NOME_ARQUIVO_TXT
CAMINHO_SAVE_ESTRUTURADO = os.path.join(PASTA_DRIVE, 'dados_estruturados.csv')
CAMINHO_COMPLETO_ENTRADA = os.path.join(PASTA_DRIVE, ARQUIVO_ENTRADA_TXT)

# Define o nome "curto" para logs e mensagens
NOME_LOG = ARQUIVO_ENTRADA_TXT.rsplit('.', 1)[0].upper()

print(f"Caminho do arquivo de entrada definido: {CAMINHO_COMPLETO_ENTRADA}")
print(f"Caminho de saída estruturado definido: {CAMINHO_SAVE_ESTRUTURADO}")

# --- NOVO: LIMPEZA DE RESULTADOS ANTERIORES ---
if os.path.exists(CAMINHO_SAVE_ESTRUTURADO):
    try:
        os.remove(CAMINHO_SAVE_ESTRUTURADO)
        print(f"✅ Arquivo de resultados anteriores removido: {os.path.basename(CAMINHO_SAVE_ESTRUTURADO)}")
    except Exception as e:
        print(f"AVISO: Não foi possível remover o arquivo de resultados anteriores. {e}")
print("-" * 50)


# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO (COM CORREÇÃO PARA ARQUIVOS TABULARES)
# ====================================================================

def processar_dataframe(df, nome_log, caminho_saida):
    """Função auxiliar para limpeza e salvamento final do DataFrame."""
    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)
    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza inicial.")
        return False

    df.set_index('Timestamp', inplace=True)

    # Forçar tipos numéricos
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza de Temperatura/Umidade.")
        return False

    # Salvar
    df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)
    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando o padrão REGEX (para casa_noite.txt)."""
    print(f"\nIniciando estruturação (REGEX) de: {os.path.basename(caminho_entrada)}")
    padrao = re.compile(r"\[(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?Dispositivo: (?P<Dispositivo>[^,]+).*?Temp: (?P<Temp>[\d\.]+).*?Umid: (?P<Umid>[\d\.]+)")

    dados = []
    try:
        with open(caminho_entrada, 'r', encoding='utf-8') as f:
            for linha in f:
                match = padrao.search(linha)
                if match:
                    dados.append(match.groupdict())
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado: {caminho_entrada}")
        return False
    except Exception as e:
        print(f"ERRO de leitura REGEX: {e}")
        return False

    df = pd.DataFrame(dados)
    if df.empty:
        print(f"AVISO REGEX: Nenhuma linha de dados válida encontrada em {nome_log}.")
        return False

    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.rename(columns={'Temp': 'Temperatura', 'Umid': 'Umidade'}, inplace=True)

    return processar_dataframe(df, nome_log, caminho_saida)

def estruturar_log_csv(caminho_entrada, caminho_saida, nome_log):
    """
    Tenta estruturar usando separador TABULAR (\t) e, se falhar, VÍRGULA (,)
    e garante que arquivos tabulares (S1, S2, etc.) usem o NOME_LOG como ID.
    """
    print(f"\nTentando estruturação (TABULAR/CSV) de: {os.path.basename(caminho_entrada)}")

    # Ordem de tentativa: 1. Tabular (\t) para S1, S2... 2. Vírgula (,)
    separadores = [('\t', 'Tabular (\\t)'), (',', 'CSV (,)')]
    df = None
    # Usamos estes nomes para a leitura inicial
    col_names = ['Col_ID_Temporario', 'Data', 'Hora', 'Temperatura', 'Umidade']

    for sep_char, sep_nome in separadores:
        try:
            df_teste = pd.read_csv(caminho_entrada, sep=sep_char, header=None,
                                 names=col_names,
                                 engine='python', on_bad_lines='skip')

            if df_teste.shape[1] >= 5 and df_teste.shape[0] > 0:
                df = df_teste.iloc[:, :5].copy()
                print(f"INFO: Usando formato delimitado ({sep_nome}) para {nome_log}")

                # CORREÇÃO CRÍTICA: Se a coluna 1 for numérica (contador), força o ID do dispositivo para o nome do arquivo.
                # Isso impede a análise por linha (S1_1, S1_2...).
                if df['Col_ID_Temporario'].astype(str).str.isnumeric().all():
                     df['Dispositivo'] = nome_log # Define o ID único
                else:
                     # Se não for numérica, usa a coluna original (pode ser o caso de outros CSVs)
                     df.rename(columns={'Col_ID_Temporario': 'Dispositivo'}, inplace=True)

                break

        except Exception:
            continue

    if df is None or df.empty:
        print(f"AVISO TABULAR/CSV: O arquivo não pôde ser lido corretamente com Tabulação ou Vírgula.")
        return False

    # Renomeia as colunas restantes para o padrão
    df.rename(columns={'Col_ID_Temporario': 'Dispositivo'}, inplace=True, errors='ignore')

    # Converte Data e Hora para Timestamp
    df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

    return processar_dataframe(df, nome_log, caminho_saida)


# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

if not estruturar_log_regex(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG):
    estruturar_log_csv(CAMINHO_COMPLETO_ENTRADA, CAMINHO_SAVE_ESTRUTURADO, NOME_LOG)


# ====================================================================
# 3. Carregamento Final (Inalterado)
# ====================================================================

df_dados = pd.DataFrame()

try:
    df_dados = pd.read_csv(CAMINHO_SAVE_ESTRUTURADO, index_col=0, parse_dates=True)
    df_dados.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_dados['Temp'] = pd.to_numeric(df_dados['Temp'], errors='coerce')
    df_dados['Umid'] = pd.to_numeric(df_dados['Umid'], errors='coerce')
    df_dados.dropna(subset=['Temp', 'Umid'], inplace=True)

except FileNotFoundError:
    print("\nERRO FATAL: O arquivo estruturado não foi encontrado. A estruturação falhou.")
    exit()
except Exception:
    pass

if df_dados.empty:
    print(f"\nERRO FATAL: O DataFrame {NOME_LOG} está vazio. Verifique o formato do arquivo TXT.")
    exit()
else:
    print(f"\nDataFrame {NOME_LOG} carregado com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Cálculo das Médias ARITMÉTICAS (MODIFICADO: Groupby usa apenas o Dispositivo)
# ====================================================================

resultados_analise = []
# Agora o Groupby funciona corretamente para ambos os formatos
grupos_dispositivo = df_dados.groupby('Dispositivo')

print("\n--- ⏳ Calculando Médias ARITMÉTICAS (Simples) ---")

for dispositivo, df_grupo in grupos_dispositivo:
    if len(df_grupo) < 1:
        continue

    media_temp = df_grupo['Temp'].mean()
    media_umid = df_grupo['Umid'].mean()

    resultados_analise.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Aritmetica': media_temp,
        'Umid_Média_Aritmetica': media_umid,
        'Total_Pontos': len(df_grupo)
    })

df_resultados = pd.DataFrame(resultados_analise).set_index('Dispositivo')
df_resultados.dropna(inplace=True)


# ====================================================================
# 5. Apresentação dos Resultados (Inalterado)
# ====================================================================

# 1. Cálculo da Média Aritmética GERAL
media_temp_geral = df_dados['Temp'].mean()
media_umid_geral = df_dados['Umid'].mean()

# 2. Apresentação do Resultado FINAL (Geral)
print("\n" + "="*90)
print(f"|   ANÁLISE FINAL: MÉDIA ARITMÉTICA GERAL do arquivo: {ARQUIVO_ENTRADA_TXT}   |")
print("="*90)

print("\n--- 📊 Média ARITMÉTICA por Dispositivo (Individual) ---")
# Agora, para o S1, S2, S3... ele mostrará apenas uma linha
# Para o casa_noite.txt, ele mostrará as médias por Pico W_XXXX
print(df_resultados[['Temp_Média_Aritmetica', 'Umid_Média_Aritmetica', 'Total_Pontos']].to_markdown(floatfmt=".2f"))


print("\n--- 🌟 Média ARITMÉTICA GERAL do Arquivo ---")
print(f"|   Temperatura Média GERAL: {media_temp_geral:.2f}°C")
print(f"|   Umidade Média GERAL:     {media_umid_geral:.2f}%")
print("------------------------------------------")

Montando Google Drive...
Drive já montado.
Digite o nome do arquivo TXT para análise (Ex: PicoW.txt, S1.txt): Urbano_Temperatura_local_casa_03_10
Caminho do arquivo de entrada definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/Urbano_Temperatura_local_casa_03_10.txt
Caminho de saída estruturado definido: /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao/dados_estruturados.csv
✅ Arquivo de resultados anteriores removido: dados_estruturados.csv
--------------------------------------------------

Iniciando estruturação (REGEX) de: Urbano_Temperatura_local_casa_03_10.txt
Estruturação de URBANO_TEMPERATURA_LOCAL_CASA_03_10 concluída. Salvo em: dados_estruturados.csv (Total de 1738 linhas)

DataFrame URBANO_TEMPERATURA_LOCAL_CASA_03_10 carregado com sucesso. Iniciando Análise.

--- ⏳ Calculando Médias ARITMÉTICAS (Simples) ---

|   ANÁLISE FINAL: MÉDIA ARITMÉTICA GERAL do arquivo: Urbano_Temperatura_local_casa_03_10.txt   |

--- 📊 Média ARITMÉTICA por Dispositivo (

In [1]:
from google.colab import drive
# Tenta desmontar o Drive para garantir uma nova conexão limpa
try:
    drive.flush_and_unmount()
except:
    pass
# Monta o Drive novamente. Você deve ver um link e pedir a permissão.
drive.mount('/content/drive')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
